# libgltf 代码介绍

libgltf 是 Minecraft 26.3 的 glTF 高性能渲染模组：加载 glTF 资源、材质、动画、LOD，并同时支持 Vulkan 与 OpenGL 的 GPU 渲染路径。

技术栈：Fabric / RenderPearl / Vulkan NV+EXT mesh shader / OpenGL NV+EXT mesh shader / meshoptimizer / Kotlin。

## 包结构

| 包 | 职责 |
|---|---|
| `api` | 对外加载/实例 API |
| `asset` | glTF 解析（GLB、Accessor、材质扩展） |
| `model` | 模型数据结构 |
| `material` | 材质与纹理绑定 |
| `animation` | 动画状态机与播放 |
| `lod` | LOD 策略与简化 |
| `render.gpu` | GPU 资源、meshlet 存储、能力探测 |
| `render.vulkan` | Vulkan mesh/间接绘制、配置、批量分发 |
| `render.gl` | OpenGL mesh 管线 |
| `render.feature` | GPU 提交、批处理、FeatureRenderer 接入 |
| `render.texture` | 纹理与材质贴图 |
| `integration` | 方块/实体/物品渲染集成 |

In [ ]:
import java.nio.file.Files
import java.nio.file.Paths

val root = Paths.get(System.getProperty("user.dir"), "src/main/kotlin")
val files = Files.walk(root).use { paths ->
    paths.filter(Files::isRegularFile).filter { it.toString().endsWith(".kt") }.toList()
}
println("Kotlin 文件数: ${files.size}")
println()
files.sorted().forEach { println(root.relativize(it)) }

## 渲染管线

1. `GltfSceneRenderer.submit`：每帧收集场景里的 glTF 实例，按材质/资源分组生成 `GpuSubmit`。
2. `GpuSubmitRenderer`：实现 FeatureRenderer，把提交按 `GpuBatchKey` 分组准备。
3. `GpuBatch`：每帧写入实例缓冲，选择绘制路径。
4. `MeshletDispatcher`：mesh 路径下准备 meshlet 状态；间接路径下准备命令缓冲。
5. `VulkanGpuDriver` / `GlGpuDriver`：创建 mesh 管线缓存并执行 `vkCmdDrawMeshTasksNV/EXT` 或 GL 等效调用。
6. mesh shader 读取 `MeshletStorage` 的紧凑顶点/三角形/元数据，完成剔除与顶点发射；片元阶段与 Minecraft 材质管线一致。

In [ ]:
val keyFiles = listOf(
    "render/GltfSceneRenderer.kt",
    "render/feature/GpuSubmitRenderer.kt",
    "render/feature/GpuBatch.kt",
    "render/vulkan/MeshletDispatcher.kt",
    "render/vulkan/VulkanGpuDriver.kt",
    "render/gpu/MeshletStorage.kt",
    "render/gpu/GpuMesh.kt",
    "render/vulkan/RenderConfig.kt"
)
keyFiles.forEach { rel ->
    println("===== $rel =====")
    Files.readAllLines(root.resolve(rel)).take(14).forEach(::println)
    println()
}

## 性能设计

- **紧凑顶点**：每个 meshlet 顶点 16 字节（量化 16 位位置 + oct16 法线 + fp16 UV + 8 位颜色），mesh shader 顺序读取。
- **批量分发**：NV 路径每组处理 4 个 meshlet（`meshBatchSize`），组数降 3-4 倍。
- **剔除**：球体视锥 + 紧致锥背面 + 上一帧深度遮挡（设备深度空间比较，可配置）。
- **描述符缓存**：存储描述符集按缓冲区身份跨帧缓存，稳态下零更新。
- **按后端选择**：Vulkan 用 64/64 meshlet + 批量；OpenGL 用 256/256 减少任务组。
- **配置**：`config/libgltf.properties` 可覆盖 `meshShader`、`meshBatchSize`、`occlusionCulling`、`instanceCulling`、`meshletCulling`、`groupLimit`。

## 代码审查指标

运行下面的单元格查看规模、重复类名、遗留旧前缀与 TODO 标记。

In [ ]:
data class FileInfo(val path: java.nio.file.Path, val lines: Int, val content: String)
val infos = files.map { f ->
    val text = Files.readString(f)
    FileInfo(f, text.lines().size, text)
}
println("总行数: ${infos.sumOf { it.lines }}")
println("最长的 10 个文件:")
infos.sortedByDescending { it.lines }.take(10).forEach { println("  %5d  %s".format(it.lines, root.relativize(it.path))) }

In [ ]:
val pattern = Regex("^(?:class|object|interface|enum class|data class|sealed class|abstract class)\\s+([A-Za-z0-9_]+)")
val names = infos.flatMap { info -> info.content.lines().mapNotNull { pattern.find(it)?.groupValues?.get(1) } }
val duplicates = names.groupingBy { it }.eachCount().filterValues { it > 1 }
println("顶层声明数: ${names.size}")
println("重复简单类名: ${if (duplicates.isEmpty()) "无" else duplicates}")
println("仍带 Gltf 前缀: ${names.filter { it.startsWith("Gltf") }.sorted()}")
val todo = Regex("(?i)(TODO|FIXME|XXX|HACK)")
val findings = infos.flatMap { info ->
    info.content.lines().mapIndexedNotNull { i, line ->
        if (todo.containsMatchIn(line)) "${root.relativize(info.path)}:${i + 1}: ${line.trim()}" else null
    }
}
println("TODO/FIXME: ${findings.size}")
findings.forEach { println("  $it") }